# NB08 – Generator: Stable Cascade  (`sd_cascade`)

**Run on its own Kaggle account, in parallel with the other generators.**

_1024×1024, prior 20 steps + decoder 10 steps | CPU offload (two-stage)_

### Prerequisites
- NB00, NB01, NB02 finished (repo, library, config, real images, captions exist).
- Internet **ON**, GPU **ON** (T4).
- `HF_TOKEN` secret set on this account.

### Parallel-safe
Writes only to `data/sd_cascade/`. Seed = `hash(MASTER_SEED, sd_cascade, source_real_id)`
— crashes/retries reproduce byte-identical images. Commits every 20 min. Stops at 10 K.

In [ ]:
import importlib.util, sys, subprocess, os

# ── 1. cache: weights live on the LARGE ephemeral scratch disk, NOT the 20 GB /kaggle/working ──
#    /kaggle/working is only ~20 GB and is auto-saved as notebook output → too small for these models
#    and you do NOT want 30 GB of weights committed as output. /kaggle/temp is the big scratch disk.
#    Trade-off: cache is wiped on session restart, so weights re-download — but that is fine:
#    the generated images are already safe on HF and the run resumes from the shards (reconcile_ids).
CACHE_ROOT = "/kaggle/temp/hf_cache"          # big scratch; survives within a session, not across restarts
os.environ["HF_HOME"]       = CACHE_ROOT       # transformers / diffusers / hub all read this
os.environ["HF_HUB_CACHE"]  = CACHE_ROOT + "/hub"
import importlib.util as _ilu
if _ilu.find_spec("hf_transfer") is not None:
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # accelerated downloads when the pkg is present
else:
    os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)  # avoid the "enabled but not installed" crash
os.makedirs(CACHE_ROOT, exist_ok=True)
import shutil as _sh
for _mnt, _lbl in [("/kaggle/temp","scratch (cache)"), ("/kaggle/working","working (outputs, 20 GB cap)")]:
    try:
        _t,_u,_f = _sh.disk_usage(_mnt)
        print(f"{_lbl:32s} free={_f/2**30:6.1f} GB / total={_t/2**30:6.1f} GB  ({_mnt})")
    except FileNotFoundError:
        print(f"{_lbl:32s} (path {_mnt} not present in this session)")
print("HF cache →", os.environ["HF_HOME"])

# ── 2. PIL._Ink fix – run BEFORE any import touching torchvision/transformers ──
import PIL._typing
if not hasattr(PIL._typing, "_Ink"):
    from typing import Union
    PIL._typing._Ink = Union[int, float, str, bytes, tuple]
    for _k in list(sys.modules):
        if _k in ("PIL.ImageText","PIL.ImageDraw") or _k.startswith("torchvision"):
            del sys.modules[_k]
    print("patched PIL._typing._Ink")

# ── 3. install diffusers if missing/outdated – never -U on unrelated packages ──
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
_need = []
try:
    import diffusers as _d
    _ver = tuple(int(x) for x in _d.__version__.split(".")[:2])
    if _ver < (0, 30): _need.append("diffusers>=0.30")
except ImportError:
    _need.append("diffusers>=0.30")
for _m, _p in [("safetensors","safetensors"),
               ("sentencepiece","sentencepiece"),("protobuf","protobuf")]:
    if importlib.util.find_spec(_m) is None: _need.append(_p)
if _need: _pip(*_need); print("installed:", _need)
else: print("all deps present – no upgrades made")
import diffusers, torch
print(f"diffusers {diffusers.__version__} | torch {torch.__version__} | cuda: {torch.cuda.is_available()}")

In [ ]:
REPO_ID = "Shanmuk4622/ai-detection-dataset-v2"
MODEL   = "sd_cascade"   # ← the only line that differs between NB03–NB08

import os
from huggingface_hub import hf_hub_download
def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception: pass
    for k in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass; return getpass.getpass("HF write token: ").strip()
TOKEN = load_token()
os.environ["HF_TOKEN"] = TOKEN   # authenticate EVERY download (gated FLUX needs this) + higher HF rate limits
lib = hf_hub_download(REPO_ID, "ai_detect_common.py", repo_type="dataset", token=TOKEN)
import importlib.util
spec = importlib.util.spec_from_file_location("ai_detect_common", lib)
C = importlib.util.module_from_spec(spec); spec.loader.exec_module(C)
cfg = C.read_config(REPO_ID, TOKEN)
MASTER_SEED = cfg.get("master_seed", 42)   # change master_seed in NB00 to reseed everything
print(f"library {C.PIPELINE_VERSION} loaded | MASTER_SEED={MASTER_SEED}")
assert MODEL in cfg["generators"], f"{MODEL} not in config"
g = cfg["generators"][MODEL]
print("spec:", g)

In [ ]:
import torch

# Import only the specific classes needed — AutoPipelineForText2Image cannot be imported
# because diffusers' auto_pipeline loads HunyuanDiT which requires MT5Tokenizer, which
# has been removed from newer transformers exports.
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionXLPipeline,
    FluxPipeline,
    KandinskyV22PriorPipeline,
    KandinskyV22Pipeline,
    PixArtSigmaPipeline,
    StableCascadePriorPipeline,
    StableCascadeDecoderPipeline,
)

DTYPE = torch.float16

def build_generator(model_key, g):
    mid    = g["model_id"]
    native = g["native"]
    steps  = g["steps"]
    guid   = g["guidance"]

    if model_key == "sd15":
        pipe = StableDiffusionPipeline.from_pretrained(
            mid, torch_dtype=DTYPE, safety_checker=None, requires_safety_checker=False)
        pipe.to("cuda")
        try: pipe.enable_vae_slicing()
        except Exception: pass
        pipe.set_progress_bar_config(disable=True)
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            return pipe(prompt, height=native, width=native,
                        num_inference_steps=steps, guidance_scale=guid, generator=gn).images[0]
        return gen

    if model_key == "sdxl":
        pipe = StableDiffusionXLPipeline.from_pretrained(
            mid, torch_dtype=DTYPE, use_safetensors=True, add_watermarker=False)
        pipe.to("cuda")
        try: pipe.enable_vae_slicing()
        except Exception: pass
        pipe.set_progress_bar_config(disable=True)
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            return pipe(prompt=prompt, height=native, width=native,
                        num_inference_steps=steps, guidance_scale=guid, generator=gn).images[0]
        return gen

    if model_key == "flux_schnell":
        pipe = FluxPipeline.from_pretrained(mid, torch_dtype=DTYPE)
        pipe.enable_model_cpu_offload()
        try: pipe.vae.enable_slicing(); pipe.vae.enable_tiling()
        except Exception: pass
        pipe.set_progress_bar_config(disable=True)
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            return pipe(prompt=prompt, height=native, width=native,
                        num_inference_steps=steps, guidance_scale=guid,
                        max_sequence_length=256, generator=gn).images[0]
        return gen

    if model_key == "kandinsky22":
        prior = KandinskyV22PriorPipeline.from_pretrained(
            "kandinsky-community/kandinsky-2-2-prior", torch_dtype=DTYPE)
        prior.to("cuda")
        pipe = KandinskyV22Pipeline.from_pretrained(mid, torch_dtype=DTYPE)
        pipe.to("cuda")
        prior.set_progress_bar_config(disable=True)
        pipe.set_progress_bar_config(disable=True)
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            prior_out = prior(prompt, num_inference_steps=steps,
                              guidance_scale=guid, generator=gn)
            return pipe(image_embeds=prior_out.image_embeds,
                        negative_image_embeds=prior_out.negative_image_embeds,
                        height=native, width=native,
                        num_inference_steps=steps, generator=gn).images[0]
        return gen

    if model_key == "pixart_sigma":
        pipe = PixArtSigmaPipeline.from_pretrained(mid, torch_dtype=DTYPE, use_safetensors=True)
        pipe.to("cuda")
        pipe.set_progress_bar_config(disable=True)
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            return pipe(prompt=prompt, height=native, width=native,
                        num_inference_steps=steps, guidance_scale=guid, generator=gn).images[0]
        return gen

    if model_key == "sd_cascade":
        prior   = StableCascadePriorPipeline.from_pretrained(g["prior_id"], variant="bf16", torch_dtype=DTYPE)
        decoder = StableCascadeDecoderPipeline.from_pretrained(mid, variant="bf16", torch_dtype=DTYPE)
        prior.enable_model_cpu_offload()
        decoder.enable_model_cpu_offload()
        prior.set_progress_bar_config(disable=True)
        decoder.set_progress_bar_config(disable=True)
        p_steps, d_steps = steps[0], steps[1]
        p_guid,  d_guid  = guid[0],  guid[1]
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            po = prior(prompt=prompt, height=native, width=native, negative_prompt="",
                       guidance_scale=p_guid, num_inference_steps=p_steps,
                       num_images_per_prompt=1, generator=gn)
            return decoder(image_embeddings=po.image_embeddings, prompt=prompt,
                           negative_prompt="", guidance_scale=d_guid,
                           num_inference_steps=d_steps, output_type="pil", generator=gn).images[0]
        return gen

    raise ValueError(f"unknown model_key: {model_key}")

generate = build_generator(MODEL, g)
print("pipeline ready for", MODEL)

In [ ]:
import gc, time, torch
api      = C.HfApi(token=TOKEN)
TARGET   = cfg["target_per_generator"]          # 10_000
native   = g["native"]
g_steps  = g["steps"][0]  if isinstance(g["steps"],  list) else g["steps"]
g_guid   = g["guidance"][0] if isinstance(g["guidance"], list) else g["guidance"]

# load FROZEN captions (written by NB02) → {source_real_id: caption}
captions = {}
for f in C.list_shards(REPO_ID, "captions/", TOKEN):
    local = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
    t = C.pq.read_table(local, columns=["source_real_id", "caption"])
    for sid, cap in zip(t.column("source_real_id").to_pylist(), t.column("caption").to_pylist()):
        captions[sid] = cap
assert captions, "No captions found – run NB02 first."
print(f"captions loaded: {len(captions)}")

# ══ PARALLEL WORKERS ═══════════════════════════════════════════════
# Run this SAME notebook on N Kaggle accounts at once to cut wall-clock by ~N×.
# Set NUM_WORKERS = N on every account; give each a DIFFERENT WORKER_ID in 0..N-1.
# Each image id is assigned to exactly one worker by a hash of the id, so two workers
# can NEVER make the same image (no overlap) and every image is owned by some worker
# (no gaps) — independent of start time, restarts, or how far the others have got.
# Shard files are already collision-safe: each session writes a unique uuid in its name.
import hashlib
NUM_WORKERS = 8          # how many Kaggle accounts you run in parallel (use 1 to disable)
WORKER_ID   = 0          # <<< CHANGE PER ACCOUNT: 0 on the 1st acct, 1 on the 2nd, ... up to NUM_WORKERS-1
assert 0 <= WORKER_ID < NUM_WORKERS, "WORKER_ID must be in 0..NUM_WORKERS-1"

def _owner(sid: str) -> int:
    return int(hashlib.sha256(str(sid).encode()).hexdigest(), 16) % NUM_WORKERS

# `universe` = the exact TARGET image set a single-worker run would produce (lowest-sorted ids),
# so totals stay at TARGET no matter how many captions exist.
universe = sorted(captions)[:TARGET]
mine     = [sid for sid in universe if _owner(sid) == WORKER_ID]   # this worker's fixed, disjoint slice
done     = C.reconcile_ids(REPO_ID, f"data/{MODEL}/", TOKEN)       # GLOBAL progress (all workers' shards)
todo     = [sid for sid in mine if sid not in done]                # my remaining only
print(f"worker {WORKER_ID}/{NUM_WORKERS} | my share={len(mine)} | "
      f"done(global)={len(done)} | my remaining={len(todo)} | total target={TARGET}")

writer = C.ShardWriter(api, REPO_ID, f"data/{MODEL}/",
                       commit_interval=cfg["commit_interval_seconds"],
                       max_rows=cfg["batch_size"], token=TOKEN)
accepted, failed = 0, 0
t0 = time.time()
try:
    for sid in todo:                      # process exactly this worker's slice (the slice itself caps the total)
        prompt = captions[sid]
        seed   = C.make_seed(MODEL, sid, MASTER_SEED)   # master_seed from config
        try:
            img = generate(prompt, seed)
            png = C.canonical_preprocess(img)
        except Exception as e:
            failed += 1
            if failed <= 10 or failed % 100 == 0:
                print(f"  gen failed {sid}: {type(e).__name__}: {e}")
            continue
        num = str(sid).split("_")[-1]
        row = C.empty_row()
        row.update(dict(image_id=f"{MODEL}_{num}", source_real_id=sid, label=1,
                        generator=MODEL, source_dataset=MODEL, prompt=prompt, image=png,
                        width=C.TARGET, height=C.TARGET, orig_width=native, orig_height=native,
                        gen_model_id=g["model_id"], gen_steps=int(g_steps),
                        gen_guidance=float(g_guid), gen_native_res=int(native),
                        seed=int(seed), caption_model=cfg["caption_model"],
                        pipeline_version=C.PIPELINE_VERSION,
                        sha256=C.sha256_bytes(png), created_utc=C.now_utc()))
        writer.add(row); accepted += 1
        if accepted % 100 == 0:
            elapsed = time.time() - t0
            rate    = accepted / elapsed
            eta_h   = ((len(mine) - accepted) / rate / 3600) if rate > 0 else 0
            print(f"  {MODEL} w{WORKER_ID}: {accepted}/{len(mine)} (my slice) | "
                  f"{rate:.2f} img/s | ETA {eta_h:.1f} h | failed {failed}")
            gc.collect(); torch.cuda.empty_cache()
        writer.maybe_flush()
finally:
    writer.close()
total = len(done) + accepted
print(f"done. accepted this run: {accepted} | failed: {failed} | total: {total}/{TARGET}")

## Self-check

In [ ]:
n = C.count_rows(REPO_ID, f"data/{MODEL}/", TOKEN)
print(f"{MODEL}: {n} rows  (target {cfg['target_per_generator']})")
import random
shards = C.list_shards(REPO_ID, f"data/{MODEL}/", TOKEN)
if shards:
    loc = C.hf_hub_download(REPO_ID, shards[-1], repo_type="dataset", token=TOKEN)
    t   = C.pq.read_table(loc, columns=["image","image_id","source_real_id","label","seed"])
    j   = random.randrange(t.num_rows)
    ok, why = C.png_is_canonical(t.column("image")[j].as_py())
    print("sample:", t.column("image_id")[j].as_py(),
          "←", t.column("source_real_id")[j].as_py(),
          "| label", t.column("label")[j].as_py(),
          "| seed", t.column("seed")[j].as_py(),
          "| canonical", ok, why)
print("RESULT:", "looks correct" if n > 0 else "no rows yet")